In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler


from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [2]:
train = pd.read_csv("../train.csv")
test = pd.read_csv("../test.csv")


print(f"Train Shape : {train.shape}")
print(f"Test Shape : {test.shape}")


train.head(10)

Train Shape : (15588, 14)
Test Shape : (3629, 14)


,title,year,mileage,transmission,fuel,body_color,inside_color,body_status,description,body_type,volume,engine,acceleration,price
0,هیوندای، توسان,2014,"175,000 km",اتوماتیک,9.6 لیتر در صد کیلومتر,سفید,مارون,بدون رنگ,مانیتور اندرویدی بدنه بدون لیسه کشی و روغن گیر...,crossover,2.4 لیتر,4 سیلندر,10.4 ثانیه,"2,550,000,000"
1,پراید، 131,1391,"330,000 km",دنده ای,6.7 لیتر در صد کیلومتر,سفید,سفید,بدون رنگ,ماشین شاستی سرشاستی پلمپ میباشد به شرط موتوری ...,passenger_car,1.3 لیتر,4 سیلندر یورو3 (M13),13.8 ثانیه,"215,000,000"
2,پژو، 206,1398,"132,000 km",دنده ای,6.4 لیتر در صد کیلومتر,سفید,خاکستری,بدون رنگ,خیلی تمیز فنی عالی,hatchback,1.4 لیتر,4 سیلندر TU3,14.1 ثانیه,"445,000,000"
3,تیبا، هاچ بک,1396,"103,000 km",دنده ای,6.9 لیتر در صد کیلومتر,آبی,خاکستری,بدون رنگ,ماشین دارای دوربین دنده عقب،روکش صندلی نو،زاپا...,hatchback,1.5 لیتر,4 سیلندر M15,13 ثانیه,"310,000,000"
4,پژو، پارس,1401,"37,000 km",دنده ای,7 لیتر در صد کیلومتر,خاکستری,کرم,بدون رنگ,NaN,passenger_car,1.8 لیتر,4 سیلندر XU7 پلاس,12 ثانیه,"570,000,000"
5,لکسوس، RX,2012,"330,000 km",اتوماتیک,9.9 لیتر در صد کیلومتر,سفید,مشکی,گلگیر رنگ,NaN,suv,3.5 لیتر,6 سیلندر,7.6 ثانیه,"4,990,000,000"
6,پژو، 207,1400,"95,000 km",دنده ای,6.6 لیتر در صد کیلومتر,مشکی,نوک مدادی,دو لکه رنگ,فرمان برقی گلگیر و درب عقب سمت راننده رنگ دارد...,hatchback,1.6 لیتر,4 سیلندر TU5,11.6 ثانیه,"590,000,000"
7,پژو، 206,1398,"44,000 km",دنده ای,6.4 لیتر در صد کیلومتر,سفید,مشکی,بدون رنگ,با سلام ماشین از همه لحاظ صفر و سالم و بی ایر...,hatchback,1.4 لیتر,4 سیلندر TU3,14.1 ثانیه,"480,000,000"
8,پژو، 206,1394,"150,000 km",دنده ای,6.4 لیتر در صد کیلومتر,آبی,مشکی,بدون رنگ,NaN,hatchback,1.6 لیتر,4 سیلندر TU5,11.6 ثانیه,"390,000,000"
9,پژو، 206,1388,"290,000 km",دنده ای,6.4 لیتر در صد کیلومتر,نوک مدادی,مشکی,دو لکه رنگ,موتور تعمیر اساسی شده شاسی پلمپ خیلی تمیز بدون...,hatchback,1.4 لیتر,4 سیلندر TU3,14.1 ثانیه,"280,000,000"


#   Data Types

In [3]:
train.info()


<class 'pandas.DataFrame'>
RangeIndex: 15588 entries, 0 to 15587
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   title         15588 non-null  str  
 1   year          15588 non-null  int64
 2   mileage       15588 non-null  str  
 3   transmission  15588 non-null  str  
 4   fuel          13931 non-null  str  
 5   body_color    15588 non-null  str  
 6   inside_color  15588 non-null  str  
 7   body_status   15588 non-null  str  
 8   description   13490 non-null  str  
 9   body_type     15588 non-null  str  
 10  volume        14371 non-null  str  
 11  engine        14371 non-null  str  
 12  acceleration  13752 non-null  str  
 13  price         15588 non-null  str  
dtypes: int64(1), str(13)
memory usage: 7.3 MB


#   Statistical Summary

In [4]:
train.describe()

,year
count,15588.000000
mean,1508.419810
std,238.458118
min,1355.000000
25%,1395.000000
50%,1399.000000
75%,1402.000000
max,2023.000000


#   Column names

In [5]:
train.columns

Index(['title', 'year', 'mileage', 'transmission', 'fuel', 'body_color',
       'inside_color', 'body_status', 'description', 'body_type', 'volume',
       'engine', 'acceleration', 'price'],
      dtype='str')

#   Missing Values

In [6]:
missing_values = train.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values()
missing_values

engine          1217
volume          1217
fuel            1657
acceleration    1836
description     2098
dtype: int64

#   Checking for duplicate data

In [7]:
duplicates = train.duplicated().sum()
duplicates

np.int64(0)

#   Type Analysis

In [8]:
#Evaluating manual versus automatic transmission types
train["transmission"].value_counts()

transmission
دنده ای     8277
اتوماتیک    7311
Name: count, dtype: int64

In [9]:
#Overview of Model Types
train["body_type"].value_counts()

body_type
passenger_car    7563
hatchback        3718
crossover        3057
suv               645
pickup            571
coupe              27
van                 7
Name: count, dtype: int64

# Car Title Analysis

In [10]:
train['title'].value_counts()

title
پژو، 206                   1259
پژو، 207                    842
پژو، پارس                   828
دنا، پلاس EF7               455
سمند، LX                    365
                           ... 
آلفارومئو، جولیتا             7
هایما، 7X                     7
تویوتا، یاریس صندوق دار       7
پاژن، چهار در                 7
مینی، کوپر                    6
Name: count, Length: 232, dtype: int64

# Car Brand Grouping

In [11]:
train.insert(1, 'brand', train['title'].str.split('،').str[0].str.strip())

In [12]:
train.insert(2, 'model', train['title'].str.split('،').str[1].str.strip())

In [13]:
train.head(10)

,title,brand,model,year,mileage,transmission,fuel,body_color,inside_color,body_status,description,body_type,volume,engine,acceleration,price
0,هیوندای، توسان,هیوندای,توسان,2014,"175,000 km",اتوماتیک,9.6 لیتر در صد کیلومتر,سفید,مارون,بدون رنگ,مانیتور اندرویدی بدنه بدون لیسه کشی و روغن گیر...,crossover,2.4 لیتر,4 سیلندر,10.4 ثانیه,"2,550,000,000"
1,پراید، 131,پراید,131,1391,"330,000 km",دنده ای,6.7 لیتر در صد کیلومتر,سفید,سفید,بدون رنگ,ماشین شاستی سرشاستی پلمپ میباشد به شرط موتوری ...,passenger_car,1.3 لیتر,4 سیلندر یورو3 (M13),13.8 ثانیه,"215,000,000"
2,پژو، 206,پژو,206,1398,"132,000 km",دنده ای,6.4 لیتر در صد کیلومتر,سفید,خاکستری,بدون رنگ,خیلی تمیز فنی عالی,hatchback,1.4 لیتر,4 سیلندر TU3,14.1 ثانیه,"445,000,000"
3,تیبا، هاچ بک,تیبا,هاچ بک,1396,"103,000 km",دنده ای,6.9 لیتر در صد کیلومتر,آبی,خاکستری,بدون رنگ,ماشین دارای دوربین دنده عقب،روکش صندلی نو،زاپا...,hatchback,1.5 لیتر,4 سیلندر M15,13 ثانیه,"310,000,000"
4,پژو، پارس,پژو,پارس,1401,"37,000 km",دنده ای,7 لیتر در صد کیلومتر,خاکستری,کرم,بدون رنگ,NaN,passenger_car,1.8 لیتر,4 سیلندر XU7 پلاس,12 ثانیه,"570,000,000"
5,لکسوس، RX,لکسوس,RX,2012,"330,000 km",اتوماتیک,9.9 لیتر در صد کیلومتر,سفید,مشکی,گلگیر رنگ,NaN,suv,3.5 لیتر,6 سیلندر,7.6 ثانیه,"4,990,000,000"
6,پژو، 207,پژو,207,1400,"95,000 km",دنده ای,6.6 لیتر در صد کیلومتر,مشکی,نوک مدادی,دو لکه رنگ,فرمان برقی گلگیر و درب عقب سمت راننده رنگ دارد...,hatchback,1.6 لیتر,4 سیلندر TU5,11.6 ثانیه,"590,000,000"
7,پژو، 206,پژو,206,1398,"44,000 km",دنده ای,6.4 لیتر در صد کیلومتر,سفید,مشکی,بدون رنگ,با سلام ماشین از همه لحاظ صفر و سالم و بی ایر...,hatchback,1.4 لیتر,4 سیلندر TU3,14.1 ثانیه,"480,000,000"
8,پژو، 206,پژو,206,1394,"150,000 km",دنده ای,6.4 لیتر در صد کیلومتر,آبی,مشکی,بدون رنگ,NaN,hatchback,1.6 لیتر,4 سیلندر TU5,11.6 ثانیه,"390,000,000"
9,پژو، 206,پژو,206,1388,"290,000 km",دنده ای,6.4 لیتر در صد کیلومتر,نوک مدادی,مشکی,دو لکه رنگ,موتور تعمیر اساسی شده شاسی پلمپ خیلی تمیز بدون...,hatchback,1.4 لیتر,4 سیلندر TU3,14.1 ثانیه,"280,000,000"


# Year Analysis

In [14]:
train['year'].value_counts()

year
1402    1541
1401    1475
1397    1310
1396    1116
1400    1033
        ... 
1370       1
1971       1
1368       1
1355       1
1996       1
Name: count, Length: 77, dtype: int64

# Year Standardization

In [15]:
shamsi = (train['year'] >= 1300)&(train['year'] <= 1499)
miladi = (train['year'] >= 1900)&(train['year'] <= 2026)
#miladi = shamsi + 621
train.loc[shamsi, 'year'] = train.loc[shamsi, 'year'] + 621
train['year'].value_counts()

year
2017    1656
2023    1543
2018    1514
2022    1476
2016    1148
2021    1033
2015     807
2019     784
2020     747
2014     739
2010     559
2011     532
2012     497
2008     401
2009     361
2007     326
2013     324
2006     255
2005     207
2024     200
2004     152
2003     106
2002      64
2001      39
1998      25
1999      11
1997      10
2000      10
1993       9
1994       8
1996       8
1992       7
1991       5
1995       5
1977       3
1974       3
1984       2
1976       2
1980       1
1987       1
1978       1
1975       1
1983       1
1988       1
1979       1
1985       1
1971       1
1989       1
Name: count, dtype: int64

# Mileage Standardization

In [16]:
print(train['mileage'].head(10))
print(train['mileage'].dtype)

0    175,000 km
1    330,000 km
2    132,000 km
3    103,000 km
4     37,000 km
5    330,000 km
6     95,000 km
7     44,000 km
8    150,000 km
9    290,000 km
Name: mileage, dtype: str
str


In [17]:
train['mileage'] = train['mileage'].str.replace(',','').str.replace('km', '').str.strip()
train.head(10)

,title,brand,model,year,mileage,transmission,fuel,body_color,inside_color,body_status,description,body_type,volume,engine,acceleration,price
0,هیوندای، توسان,هیوندای,توسان,2014,175000,اتوماتیک,9.6 لیتر در صد کیلومتر,سفید,مارون,بدون رنگ,مانیتور اندرویدی بدنه بدون لیسه کشی و روغن گیر...,crossover,2.4 لیتر,4 سیلندر,10.4 ثانیه,"2,550,000,000"
1,پراید، 131,پراید,131,2012,330000,دنده ای,6.7 لیتر در صد کیلومتر,سفید,سفید,بدون رنگ,ماشین شاستی سرشاستی پلمپ میباشد به شرط موتوری ...,passenger_car,1.3 لیتر,4 سیلندر یورو3 (M13),13.8 ثانیه,"215,000,000"
2,پژو، 206,پژو,206,2019,132000,دنده ای,6.4 لیتر در صد کیلومتر,سفید,خاکستری,بدون رنگ,خیلی تمیز فنی عالی,hatchback,1.4 لیتر,4 سیلندر TU3,14.1 ثانیه,"445,000,000"
3,تیبا، هاچ بک,تیبا,هاچ بک,2017,103000,دنده ای,6.9 لیتر در صد کیلومتر,آبی,خاکستری,بدون رنگ,ماشین دارای دوربین دنده عقب،روکش صندلی نو،زاپا...,hatchback,1.5 لیتر,4 سیلندر M15,13 ثانیه,"310,000,000"
4,پژو، پارس,پژو,پارس,2022,37000,دنده ای,7 لیتر در صد کیلومتر,خاکستری,کرم,بدون رنگ,NaN,passenger_car,1.8 لیتر,4 سیلندر XU7 پلاس,12 ثانیه,"570,000,000"
5,لکسوس، RX,لکسوس,RX,2012,330000,اتوماتیک,9.9 لیتر در صد کیلومتر,سفید,مشکی,گلگیر رنگ,NaN,suv,3.5 لیتر,6 سیلندر,7.6 ثانیه,"4,990,000,000"
6,پژو، 207,پژو,207,2021,95000,دنده ای,6.6 لیتر در صد کیلومتر,مشکی,نوک مدادی,دو لکه رنگ,فرمان برقی گلگیر و درب عقب سمت راننده رنگ دارد...,hatchback,1.6 لیتر,4 سیلندر TU5,11.6 ثانیه,"590,000,000"
7,پژو، 206,پژو,206,2019,44000,دنده ای,6.4 لیتر در صد کیلومتر,سفید,مشکی,بدون رنگ,با سلام ماشین از همه لحاظ صفر و سالم و بی ایر...,hatchback,1.4 لیتر,4 سیلندر TU3,14.1 ثانیه,"480,000,000"
8,پژو، 206,پژو,206,2015,150000,دنده ای,6.4 لیتر در صد کیلومتر,آبی,مشکی,بدون رنگ,NaN,hatchback,1.6 لیتر,4 سیلندر TU5,11.6 ثانیه,"390,000,000"
9,پژو، 206,پژو,206,2009,290000,دنده ای,6.4 لیتر در صد کیلومتر,نوک مدادی,مشکی,دو لکه رنگ,موتور تعمیر اساسی شده شاسی پلمپ خیلی تمیز بدون...,hatchback,1.4 لیتر,4 سیلندر TU3,14.1 ثانیه,"280,000,000"


In [18]:
train.loc[
    pd.to_numeric(train['mileage'], errors='coerce').isna(),
    'mileage'
].value_counts()


mileage
کارکرده    272
Name: count, dtype: int64

In [19]:
train['mileage'] = train['mileage'].replace('کارکرده', np.nan)

In [20]:
train['mileage'] = pd.to_numeric(train['mileage'])
train['mileage']

0        175000.0
1        330000.0
2        132000.0
3        103000.0
4         37000.0
           ...   
15583    230000.0
15584     50000.0
15585      9500.0
15586    178000.0
15587     37000.0
Name: mileage, Length: 15588, dtype: float64

# Fuel Standardization

In [21]:
train['fuel'].value_counts()

fuel
6.4 لیتر در صد کیلومتر     1696
6.9 لیتر در صد کیلومتر     1037
7 لیتر در صد کیلومتر        874
6.6 لیتر در صد کیلومتر      858
7.7 لیتر در صد کیلومتر      601
                           ... 
14.5 لیتر در صد کیلومتر       2
5.4 لیتر در صد کیلومتر        2
11.4 لیتر در صد کیلومتر       2
4.8 لیتر در صد کیلومتر        2
10.6 لیتر در صد کیلومتر       1
Name: count, Length: 78, dtype: int64

In [22]:
train['fuel'] = train['fuel'].str.replace('لیتر در صد کیلومتر', '')
train['fuel']

0        9.6 
1        6.7 
2        6.4 
3        6.9 
4          7 
         ... 
15583    6.9 
15584    6.9 
15585    6.6 
15586      7 
15587    7.3 
Name: fuel, Length: 15588, dtype: str

In [23]:
train['fuel'] = pd.to_numeric(train['fuel'])
train['fuel']

0        9.6
1        6.7
2        6.4
3        6.9
4        7.0
        ... 
15583    6.9
15584    6.9
15585    6.6
15586    7.0
15587    7.3
Name: fuel, Length: 15588, dtype: float64

In [24]:
train.head(10)

,title,brand,model,year,mileage,transmission,fuel,body_color,inside_color,body_status,description,body_type,volume,engine,acceleration,price
0,هیوندای، توسان,هیوندای,توسان,2014,175000.0,اتوماتیک,9.6,سفید,مارون,بدون رنگ,مانیتور اندرویدی بدنه بدون لیسه کشی و روغن گیر...,crossover,2.4 لیتر,4 سیلندر,10.4 ثانیه,"2,550,000,000"
1,پراید، 131,پراید,131,2012,330000.0,دنده ای,6.7,سفید,سفید,بدون رنگ,ماشین شاستی سرشاستی پلمپ میباشد به شرط موتوری ...,passenger_car,1.3 لیتر,4 سیلندر یورو3 (M13),13.8 ثانیه,"215,000,000"
2,پژو، 206,پژو,206,2019,132000.0,دنده ای,6.4,سفید,خاکستری,بدون رنگ,خیلی تمیز فنی عالی,hatchback,1.4 لیتر,4 سیلندر TU3,14.1 ثانیه,"445,000,000"
3,تیبا، هاچ بک,تیبا,هاچ بک,2017,103000.0,دنده ای,6.9,آبی,خاکستری,بدون رنگ,ماشین دارای دوربین دنده عقب،روکش صندلی نو،زاپا...,hatchback,1.5 لیتر,4 سیلندر M15,13 ثانیه,"310,000,000"
4,پژو، پارس,پژو,پارس,2022,37000.0,دنده ای,7.0,خاکستری,کرم,بدون رنگ,NaN,passenger_car,1.8 لیتر,4 سیلندر XU7 پلاس,12 ثانیه,"570,000,000"
5,لکسوس، RX,لکسوس,RX,2012,330000.0,اتوماتیک,9.9,سفید,مشکی,گلگیر رنگ,NaN,suv,3.5 لیتر,6 سیلندر,7.6 ثانیه,"4,990,000,000"
6,پژو، 207,پژو,207,2021,95000.0,دنده ای,6.6,مشکی,نوک مدادی,دو لکه رنگ,فرمان برقی گلگیر و درب عقب سمت راننده رنگ دارد...,hatchback,1.6 لیتر,4 سیلندر TU5,11.6 ثانیه,"590,000,000"
7,پژو، 206,پژو,206,2019,44000.0,دنده ای,6.4,سفید,مشکی,بدون رنگ,با سلام ماشین از همه لحاظ صفر و سالم و بی ایر...,hatchback,1.4 لیتر,4 سیلندر TU3,14.1 ثانیه,"480,000,000"
8,پژو، 206,پژو,206,2015,150000.0,دنده ای,6.4,آبی,مشکی,بدون رنگ,NaN,hatchback,1.6 لیتر,4 سیلندر TU5,11.6 ثانیه,"390,000,000"
9,پژو، 206,پژو,206,2009,290000.0,دنده ای,6.4,نوک مدادی,مشکی,دو لکه رنگ,موتور تعمیر اساسی شده شاسی پلمپ خیلی تمیز بدون...,hatchback,1.4 لیتر,4 سیلندر TU3,14.1 ثانیه,"280,000,000"


# Volume Standardization

In [25]:
train['volume'].value_counts()

volume
1.6 لیتر     3674
1.5 لیتر     2647
2 لیتر       1630
1.8 لیتر     1497
1.7 لیتر      992
2.4 لیتر      987
1.4 لیتر      933
1.3 لیتر      502
2.0 لیتر      337
2.5 لیتر      212
2.7 لیتر      110
3.5 لیتر      106
4 لیتر        102
3 لیتر         98
3.3 لیتر       94
1.0 لیتر       90
3.8 لیتر       75
1.65 لیتر      71
1.1 لیتر       44
1.2 لیتر       39
1 لیتر         26
0.8 لیتر       23
2.8 لیتر       19
4.6 لیتر       18
4.8 لیتر       13
1.9 لیتر       11
1.62 لیتر      10
3.6 لیتر        7
4.7 لیتر        2
4.4 لیتر        2
Name: count, dtype: int64

In [26]:
train['volume'] = train['volume'].str.replace('لیتر','')
train['volume']

0        2.4 
1        1.3 
2        1.4 
3        1.5 
4        1.8 
         ... 
15583    1.6 
15584    1.6 
15585    1.6 
15586      2 
15587    1.6 
Name: volume, Length: 15588, dtype: str

In [27]:
train['volume'] = pd.to_numeric(train['volume'])
train['volume']

0        2.4
1        1.3
2        1.4
3        1.5
4        1.8
        ... 
15583    1.6
15584    1.6
15585    1.6
15586    2.0
15587    1.6
Name: volume, Length: 15588, dtype: float64

# Acceleration Standardization

In [28]:
train['acceleration'].value_counts()

acceleration
11.6 ثانیه    1122
13 ثانیه       993
14.1 ثانیه     865
12 ثانیه       836
10.5 ثانیه     797
              ... 
6.8 ثانیه        2
5.3 ثانیه        2
14.8 ثانیه       1
13.9 ثانیه       1
6 ثانیه          1
Name: count, Length: 93, dtype: int64

In [29]:
train['acceleration'] = train['acceleration'].str.replace('ثانیه','')
train['acceleration']

0        10.4 
1        13.8 
2        14.1 
3          13 
4          12 
         ...  
15583      12 
15584      12 
15585    11.6 
15586     9.5 
15587      13 
Name: acceleration, Length: 15588, dtype: str

In [30]:
train.loc[train['acceleration'].str.contains('/', na=False), 'acceleration'].value_counts()

acceleration
12/8     35
Name: count, dtype: int64

In [31]:
train['acceleration'] = train['acceleration'].astype(str).str.replace('/', '')

In [32]:
train['acceleration'] = pd.to_numeric(train['acceleration'])
train['acceleration']

0        10.4
1        13.8
2        14.1
3        13.0
4        12.0
         ... 
15583    12.0
15584    12.0
15585    11.6
15586     9.5
15587    13.0
Name: acceleration, Length: 15588, dtype: float64

# Engine Feature Extraction(Cylinder Count)

In [33]:
train['engine'].value_counts()

engine
4 سیلندر                        3262
4 سیلندر TU5                    1874
4 سیلندر توربوشارژ               987
4 سیلندر TU3                     889
4 سیلندر EF7                     537
                                ... 
4 سیلندر توربوشارژ (9NR-FTS)       1
6 سیلندر GDi                       1
4 سیلندر (GAMMA)                   1
6 سیلندر توربوشارژ                 1
4 سیلندر KA24 انژکتوری             1
Name: count, Length: 108, dtype: int64

In [34]:
train['engine'].str.extract(r'(\d+)\s*سیلندر')[0].value_counts()

0
4    13635
6      548
3      141
8       35
Name: count, dtype: int64

In [35]:
train.insert(14, 'cylinders', train['engine'].str.extract(r'(\d+)').astype(float))
train.head(10)

,title,brand,model,year,mileage,transmission,fuel,body_color,inside_color,body_status,description,body_type,volume,engine,cylinders,acceleration,price
0,هیوندای، توسان,هیوندای,توسان,2014,175000.0,اتوماتیک,9.6,سفید,مارون,بدون رنگ,مانیتور اندرویدی بدنه بدون لیسه کشی و روغن گیر...,crossover,2.4,4 سیلندر,4.0,10.4,"2,550,000,000"
1,پراید، 131,پراید,131,2012,330000.0,دنده ای,6.7,سفید,سفید,بدون رنگ,ماشین شاستی سرشاستی پلمپ میباشد به شرط موتوری ...,passenger_car,1.3,4 سیلندر یورو3 (M13),4.0,13.8,"215,000,000"
2,پژو، 206,پژو,206,2019,132000.0,دنده ای,6.4,سفید,خاکستری,بدون رنگ,خیلی تمیز فنی عالی,hatchback,1.4,4 سیلندر TU3,4.0,14.1,"445,000,000"
3,تیبا، هاچ بک,تیبا,هاچ بک,2017,103000.0,دنده ای,6.9,آبی,خاکستری,بدون رنگ,ماشین دارای دوربین دنده عقب،روکش صندلی نو،زاپا...,hatchback,1.5,4 سیلندر M15,4.0,13.0,"310,000,000"
4,پژو، پارس,پژو,پارس,2022,37000.0,دنده ای,7.0,خاکستری,کرم,بدون رنگ,NaN,passenger_car,1.8,4 سیلندر XU7 پلاس,4.0,12.0,"570,000,000"
5,لکسوس، RX,لکسوس,RX,2012,330000.0,اتوماتیک,9.9,سفید,مشکی,گلگیر رنگ,NaN,suv,3.5,6 سیلندر,6.0,7.6,"4,990,000,000"
6,پژو، 207,پژو,207,2021,95000.0,دنده ای,6.6,مشکی,نوک مدادی,دو لکه رنگ,فرمان برقی گلگیر و درب عقب سمت راننده رنگ دارد...,hatchback,1.6,4 سیلندر TU5,4.0,11.6,"590,000,000"
7,پژو، 206,پژو,206,2019,44000.0,دنده ای,6.4,سفید,مشکی,بدون رنگ,با سلام ماشین از همه لحاظ صفر و سالم و بی ایر...,hatchback,1.4,4 سیلندر TU3,4.0,14.1,"480,000,000"
8,پژو، 206,پژو,206,2015,150000.0,دنده ای,6.4,آبی,مشکی,بدون رنگ,NaN,hatchback,1.6,4 سیلندر TU5,4.0,11.6,"390,000,000"
9,پژو، 206,پژو,206,2009,290000.0,دنده ای,6.4,نوک مدادی,مشکی,دو لکه رنگ,موتور تعمیر اساسی شده شاسی پلمپ خیلی تمیز بدون...,hatchback,1.4,4 سیلندر TU3,4.0,14.1,"280,000,000"


In [36]:
train['cylinders'].isna().sum()

np.int64(1229)

In [37]:
train.loc[train['cylinders'].isna(), 'engine'].value_counts()

engine
چهار سیلندر    12
Name: count, dtype: int64

In [38]:
train.loc[train['engine'] == 'چهار سیلندر', 'cylinders'] = 4

In [39]:
train.loc[train['cylinders'].isna(), 'engine'].value_counts()

Series([], Name: count, dtype: int64)

# Engine Type Extraction

In [40]:
train['engine'].value_counts().head(30)

engine
4 سیلندر                       3262
4 سیلندر TU5                   1874
4 سیلندر توربوشارژ              987
4 سیلندر TU3                    889
4 سیلندر EF7                    537
4 سیلندر M15                    525
4 سیلندر XU7                    523
6 سیلندر                        497
4 سیلندر M15 یورو4              430
4 سیلندر توربوشارژ EF7          320
4 سیلندر یورو4 بهینه (M13)      312
4 سیلندر MPI                    230
4 سیلندر L3                     172
4 سیلندر TU5 پلاس               157
4 سیلندر توربوشارژ (M15T)       155
4 سیلندر توربوشارژ TGDi         151
4 سیلندر توربوشارژ (N20B20)     145
4 سیلندر توربوشارژ (4T15C)      139
4 سیلندر توربوشارژ EP6          130
4 سیلندر XU7 پلاس               125
4 سیلندر GDI                    109
4 سیلندر GDi: سفارشی            108
4 سیلندر MPi                     99
4 سیلندر توربوشارژ یورو5         99
3 سیلندر توربوشارژ               92
4 سیلندر TU5 یورو5               91
4 سیلندر با موتور الکتریکی       90
4 سیلندر (BM15L)     

In [41]:
train['engine'].str.replace(r'^\d+\s*سیلندر\s*', '', regex=True).value_counts().head(30)

engine
                      3838
TU5                   1874
توربوشارژ             1082
TU3                    889
EF7                    537
M15                    525
XU7                    523
M15 یورو4              430
توربوشارژ EF7          320
یورو4 بهینه (M13)      312
MPI                    230
L3                     172
TU5 پلاس               157
توربوشارژ (M15T)       155
توربوشارژ TGDi         151
توربوشارژ (N20B20)     145
توربوشارژ (4T15C)      139
MPi                    133
توربوشارژ EP6          130
XU7 پلاس               125
GDI                    109
GDi: سفارشی            108
توربوشارژ یورو5         99
TU5 یورو5               91
با موتور الکتریکی       90
(BM15L)                 86
TU5P                    86
یورو3 (M13)             85
TU5 یورو4               83
XU7 دوگانه سوز          82
Name: count, dtype: int64

In [42]:
train.insert(15, 'engine_type', train['engine'].str.replace(r'^\d+\s*سیلندر\s*', '', regex=True).str.strip())

In [43]:
train['engine_type'].value_counts()

engine_type
                       3838
TU5                    1874
توربوشارژ              1082
TU3                     889
EF7                     537
                       ... 
یورو5                     2
یورو4 (4G69)              2
توربوشارژ (9NR-FTS)       1
(GAMMA)                   1
KA24 انژکتوری             1
Name: count, Length: 100, dtype: int64

In [44]:
train['engine_type'] = train['engine_type'].replace('', np.nan)

In [45]:
train['engine_type'].isna().sum()

np.int64(5055)

In [46]:
train['engine_type'].value_counts()

engine_type
TU5                    1874
توربوشارژ              1082
TU3                     889
EF7                     537
M15                     525
                       ... 
یورو5                     2
یورو4 (4G69)              2
توربوشارژ (9NR-FTS)       1
(GAMMA)                   1
KA24 انژکتوری             1
Name: count, Length: 99, dtype: int64

# Engine Type and Turbo Feature Extraction

In [47]:
train['engine_type']

0                NaN
1        یورو3 (M13)
2                TU3
3                M15
4           XU7 پلاس
            ...     
15583      TU5 یورو4
15584      TU5 یورو5
15585            TU5
15586           XU10
15587       TU5 پلاس
Name: engine_type, Length: 15588, dtype: str

In [48]:
train.insert(15, 'engine_model', np.nan)
train.head(5)

,title,brand,model,year,mileage,transmission,fuel,body_color,inside_color,body_status,description,body_type,volume,engine,cylinders,engine_model,engine_type,acceleration,price
0,هیوندای، توسان,هیوندای,توسان,2014,175000.0,اتوماتیک,9.6,سفید,مارون,بدون رنگ,مانیتور اندرویدی بدنه بدون لیسه کشی و روغن گیر...,crossover,2.4,4 سیلندر,4.0,NaN,NaN,10.4,"2,550,000,000"
1,پراید، 131,پراید,131,2012,330000.0,دنده ای,6.7,سفید,سفید,بدون رنگ,ماشین شاستی سرشاستی پلمپ میباشد به شرط موتوری ...,passenger_car,1.3,4 سیلندر یورو3 (M13),4.0,NaN,یورو3 (M13),13.8,"215,000,000"
2,پژو، 206,پژو,206,2019,132000.0,دنده ای,6.4,سفید,خاکستری,بدون رنگ,خیلی تمیز فنی عالی,hatchback,1.4,4 سیلندر TU3,4.0,NaN,TU3,14.1,"445,000,000"
3,تیبا، هاچ بک,تیبا,هاچ بک,2017,103000.0,دنده ای,6.9,آبی,خاکستری,بدون رنگ,ماشین دارای دوربین دنده عقب،روکش صندلی نو،زاپا...,hatchback,1.5,4 سیلندر M15,4.0,NaN,M15,13.0,"310,000,000"
4,پژو، پارس,پژو,پارس,2022,37000.0,دنده ای,7.0,خاکستری,کرم,بدون رنگ,NaN,passenger_car,1.8,4 سیلندر XU7 پلاس,4.0,NaN,XU7 پلاس,12.0,"570,000,000"


In [49]:
train['engine_type'].value_counts().head(20)

engine_type
TU5                   1874
توربوشارژ             1082
TU3                    889
EF7                    537
M15                    525
XU7                    523
M15 یورو4              430
توربوشارژ EF7          320
یورو4 بهینه (M13)      312
MPI                    230
L3                     172
TU5 پلاس               157
توربوشارژ (M15T)       155
توربوشارژ TGDi         151
توربوشارژ (N20B20)     145
توربوشارژ (4T15C)      139
MPi                    133
توربوشارژ EP6          130
XU7 پلاس               125
GDI                    109
Name: count, dtype: int64

In [50]:
train['engine_model'] = train['engine_type'].str.extract(r'\b(TU5|TU3|EF7|M15|XU7)\b')
train.head(5)

,title,brand,model,year,mileage,transmission,fuel,body_color,inside_color,body_status,description,body_type,volume,engine,cylinders,engine_model,engine_type,acceleration,price
0,هیوندای، توسان,هیوندای,توسان,2014,175000.0,اتوماتیک,9.6,سفید,مارون,بدون رنگ,مانیتور اندرویدی بدنه بدون لیسه کشی و روغن گیر...,crossover,2.4,4 سیلندر,4.0,NaN,NaN,10.4,"2,550,000,000"
1,پراید، 131,پراید,131,2012,330000.0,دنده ای,6.7,سفید,سفید,بدون رنگ,ماشین شاستی سرشاستی پلمپ میباشد به شرط موتوری ...,passenger_car,1.3,4 سیلندر یورو3 (M13),4.0,NaN,یورو3 (M13),13.8,"215,000,000"
2,پژو، 206,پژو,206,2019,132000.0,دنده ای,6.4,سفید,خاکستری,بدون رنگ,خیلی تمیز فنی عالی,hatchback,1.4,4 سیلندر TU3,4.0,TU3,TU3,14.1,"445,000,000"
3,تیبا، هاچ بک,تیبا,هاچ بک,2017,103000.0,دنده ای,6.9,آبی,خاکستری,بدون رنگ,ماشین دارای دوربین دنده عقب،روکش صندلی نو،زاپا...,hatchback,1.5,4 سیلندر M15,4.0,M15,M15,13.0,"310,000,000"
4,پژو، پارس,پژو,پارس,2022,37000.0,دنده ای,7.0,خاکستری,کرم,بدون رنگ,NaN,passenger_car,1.8,4 سیلندر XU7 پلاس,4.0,XU7,XU7 پلاس,12.0,"570,000,000"


In [51]:
train['engine_type'].value_counts().iloc[20:40]

engine_type
GDi: سفارشی               108
توربوشارژ یورو5            99
TU5 یورو5                  91
با موتور الکتریکی          90
(BM15L)                    86
TU5P                       86
یورو3 (M13)                85
TU5 یورو4                  83
XU7 دوگانه سوز             82
MPi (موتور GDI سفارشی)     74
توربوشارژ (4J16)           69
BM15L                      68
یورو3                      67
توربوشارژ بهینه شده        62
توربوشارژ (4A95TD)         60
(BM16LB)                   54
توربو شارژ                 49
OHVG2 دوگانه سوز           48
(M13)                      48
دوگانه سوز                 47
Name: count, dtype: int64

In [52]:
train['engine_type'].value_counts().iloc[40:100]

engine_type
یورو پنج (DAE481Q)               46
توربوشارژ TCe200                 45
XU10                             43
GDi با موتورالکتریکی             41
(4G15C)                          40
TU5JP4                           36
توربوشارژ (4K22)                 34
TU5P یورو5                       34
دوگانه سوز (OHV G2)              31
Z24 کاربراتوری                   31
توربوشارژ (4J20)                 30
توربوشارژ (SFG15T)               27
توربوشارژ (4G20T15)              22
توربوشارژ و سوپرشارژ T6          22
MPFI                             22
GDi                              21
OHV2                             21
توربوشارژ TCe120                 19
توربوشارژ (4J15T)                18
M15 دوگانه سوز                   18
M15i بهبودیافته                  17
توربوشارژ THP165                 17
یورو 5 (4A92)                    17
EF7 یورو 4                       16
P40 کاربراتوری                   15
XU7J4P                           15
XU7P                             14
دوگانه سوز TU5JP

In [53]:
train['engine_model'] = train['engine_type'].str.extract(
    r'\b(TU5|TU3|EF7|M15|XU7|M13|BM15L|TU5P|4J16|4A95TD|BM16LB|'
    r'XU10|4G15C|TU5JP4|4K22|4J20|SFG15T|4G20T15|Z24|4J15T|'
    r'THP165|4A92|XU7J4P|XU7P|4G69|XUM|4GC16D|G4GC|GA16|'
    r'GA16YF|ZG24|2NZ|B48B20A|B48B20B|9NR-FTS|KA24|M15T|'
    r'N20B20|4T15C|EP6|TCe200|TCe120|T6|DG15T|DAE481Q|'
    r'OHVG2|P40|L3|OHV2|M15i|TU5JP4G|GAMMA)\b'
)
train.head(10)

,title,brand,model,year,mileage,transmission,fuel,body_color,inside_color,body_status,description,body_type,volume,engine,cylinders,engine_model,engine_type,acceleration,price
0,هیوندای، توسان,هیوندای,توسان,2014,175000.0,اتوماتیک,9.6,سفید,مارون,بدون رنگ,مانیتور اندرویدی بدنه بدون لیسه کشی و روغن گیر...,crossover,2.4,4 سیلندر,4.0,NaN,NaN,10.4,"2,550,000,000"
1,پراید، 131,پراید,131,2012,330000.0,دنده ای,6.7,سفید,سفید,بدون رنگ,ماشین شاستی سرشاستی پلمپ میباشد به شرط موتوری ...,passenger_car,1.3,4 سیلندر یورو3 (M13),4.0,M13,یورو3 (M13),13.8,"215,000,000"
2,پژو، 206,پژو,206,2019,132000.0,دنده ای,6.4,سفید,خاکستری,بدون رنگ,خیلی تمیز فنی عالی,hatchback,1.4,4 سیلندر TU3,4.0,TU3,TU3,14.1,"445,000,000"
3,تیبا، هاچ بک,تیبا,هاچ بک,2017,103000.0,دنده ای,6.9,آبی,خاکستری,بدون رنگ,ماشین دارای دوربین دنده عقب،روکش صندلی نو،زاپا...,hatchback,1.5,4 سیلندر M15,4.0,M15,M15,13.0,"310,000,000"
4,پژو، پارس,پژو,پارس,2022,37000.0,دنده ای,7.0,خاکستری,کرم,بدون رنگ,NaN,passenger_car,1.8,4 سیلندر XU7 پلاس,4.0,XU7,XU7 پلاس,12.0,"570,000,000"
5,لکسوس، RX,لکسوس,RX,2012,330000.0,اتوماتیک,9.9,سفید,مشکی,گلگیر رنگ,NaN,suv,3.5,6 سیلندر,6.0,NaN,NaN,7.6,"4,990,000,000"
6,پژو، 207,پژو,207,2021,95000.0,دنده ای,6.6,مشکی,نوک مدادی,دو لکه رنگ,فرمان برقی گلگیر و درب عقب سمت راننده رنگ دارد...,hatchback,1.6,4 سیلندر TU5,4.0,TU5,TU5,11.6,"590,000,000"
7,پژو، 206,پژو,206,2019,44000.0,دنده ای,6.4,سفید,مشکی,بدون رنگ,با سلام ماشین از همه لحاظ صفر و سالم و بی ایر...,hatchback,1.4,4 سیلندر TU3,4.0,TU3,TU3,14.1,"480,000,000"
8,پژو، 206,پژو,206,2015,150000.0,دنده ای,6.4,آبی,مشکی,بدون رنگ,NaN,hatchback,1.6,4 سیلندر TU5,4.0,TU5,TU5,11.6,"390,000,000"
9,پژو، 206,پژو,206,2009,290000.0,دنده ای,6.4,نوک مدادی,مشکی,دو لکه رنگ,موتور تعمیر اساسی شده شاسی پلمپ خیلی تمیز بدون...,hatchback,1.4,4 سیلندر TU3,4.0,TU3,TU3,14.1,"280,000,000"


In [54]:
train['engine_model'].value_counts()

engine_model
TU5        2205
M15         973
EF7         890
TU3         889
XU7         730
M13         445
L3          172
M15T        155
BM15L       154
N20B20      145
4T15C       139
EP6         130
TU5P        120
4J16         69
4A95TD       60
BM16LB       54
OHVG2        50
DAE481Q      46
TCe200       45
XU10         43
4G15C        40
TU5JP4       36
4K22         34
Z24          31
4J20         30
SFG15T       27
XU7P         24
4G20T15      22
T6           22
OHV2         21
4G69         20
TCe120       19
4J15T        18
M15i         17
THP165       17
4A92         17
P40          15
XU7J4P       15
TU5JP4G      12
DG15T        12
XUM          11
4GC16D       11
G4GC         10
GA16          9
ZG24          8
GA16YF        7
2NZ           5
B48B20B       5
B48B20A       4
9NR-FTS       1
GAMMA         1
KA24          1
Name: count, dtype: int64

In [55]:
train['engine_model'].isna().sum()

np.int64(7552)

In [56]:
train[['engine', 'engine_type', 'engine_model']].sample(10)

,engine,engine_type,engine_model
10257,4 سیلندر XU7,XU7,XU7
3141,4 سیلندر TU5 یورو5,TU5 یورو5,TU5
3710,4 سیلندر TU5 یورو5,TU5 یورو5,TU5
2391,NaN,NaN,NaN
6683,4 سیلندر TU5,TU5,TU5
2269,4 سیلندر TU3,TU3,TU3
10559,4 سیلندر,NaN,NaN
12456,4 سیلندر XU7,XU7,XU7
13267,4 سیلندر TU5 یورو4,TU5 یورو4,TU5
11142,4 سیلندر,NaN,NaN


# Turbocharged Engine

In [57]:
train.insert(17, 'turbo', train['engine_type'].str.contains('توربوشارژ|توربو شارژ|تویین شارژ', na = False).astype(int))
train.head(5)
# if number is 1 = turbo, else = not turbo

,title,brand,model,year,mileage,transmission,fuel,body_color,inside_color,body_status,description,body_type,volume,engine,cylinders,engine_model,engine_type,turbo,acceleration,price
0,هیوندای، توسان,هیوندای,توسان,2014,175000.0,اتوماتیک,9.6,سفید,مارون,بدون رنگ,مانیتور اندرویدی بدنه بدون لیسه کشی و روغن گیر...,crossover,2.4,4 سیلندر,4.0,NaN,NaN,0,10.4,"2,550,000,000"
1,پراید، 131,پراید,131,2012,330000.0,دنده ای,6.7,سفید,سفید,بدون رنگ,ماشین شاستی سرشاستی پلمپ میباشد به شرط موتوری ...,passenger_car,1.3,4 سیلندر یورو3 (M13),4.0,M13,یورو3 (M13),0,13.8,"215,000,000"
2,پژو، 206,پژو,206,2019,132000.0,دنده ای,6.4,سفید,خاکستری,بدون رنگ,خیلی تمیز فنی عالی,hatchback,1.4,4 سیلندر TU3,4.0,TU3,TU3,0,14.1,"445,000,000"
3,تیبا، هاچ بک,تیبا,هاچ بک,2017,103000.0,دنده ای,6.9,آبی,خاکستری,بدون رنگ,ماشین دارای دوربین دنده عقب،روکش صندلی نو،زاپا...,hatchback,1.5,4 سیلندر M15,4.0,M15,M15,0,13.0,"310,000,000"
4,پژو، پارس,پژو,پارس,2022,37000.0,دنده ای,7.0,خاکستری,کرم,بدون رنگ,NaN,passenger_car,1.8,4 سیلندر XU7 پلاس,4.0,XU7,XU7 پلاس,0,12.0,"570,000,000"


In [58]:
train[['engine_type', 'turbo']].value_counts()

engine_type          turbo
TU5                  0        1874
توربوشارژ            1        1082
TU3                  0         889
EF7                  0         537
M15                  0         525
                              ... 
یورو5                0           2
یورو4 (4G69)         0           2
توربوشارژ (9NR-FTS)  1           1
(GAMMA)              0           1
KA24 انژکتوری        0           1
Name: count, Length: 99, dtype: int64

# Dual_Fuel System

In [59]:
train.insert(18, 'dual_fuel', train['engine_type'].str.contains('دوگانه سوز|دوگانه سوز', na = False).astype(int))
train.head(5)

,title,brand,model,year,mileage,transmission,fuel,body_color,inside_color,body_status,...,body_type,volume,engine,cylinders,engine_model,engine_type,turbo,dual_fuel,acceleration,price
0,هیوندای، توسان,هیوندای,توسان,2014,175000.0,اتوماتیک,9.6,سفید,مارون,بدون رنگ,...,crossover,2.4,4 سیلندر,4.0,NaN,NaN,0,0,10.4,"2,550,000,000"
1,پراید، 131,پراید,131,2012,330000.0,دنده ای,6.7,سفید,سفید,بدون رنگ,...,passenger_car,1.3,4 سیلندر یورو3 (M13),4.0,M13,یورو3 (M13),0,0,13.8,"215,000,000"
2,پژو، 206,پژو,206,2019,132000.0,دنده ای,6.4,سفید,خاکستری,بدون رنگ,...,hatchback,1.4,4 سیلندر TU3,4.0,TU3,TU3,0,0,14.1,"445,000,000"
3,تیبا، هاچ بک,تیبا,هاچ بک,2017,103000.0,دنده ای,6.9,آبی,خاکستری,بدون رنگ,...,hatchback,1.5,4 سیلندر M15,4.0,M15,M15,0,0,13.0,"310,000,000"
4,پژو، پارس,پژو,پارس,2022,37000.0,دنده ای,7.0,خاکستری,کرم,بدون رنگ,...,passenger_car,1.8,4 سیلندر XU7 پلاس,4.0,XU7,XU7 پلاس,0,0,12.0,"570,000,000"


In [60]:
train[['engine_type', 'dual_fuel']].value_counts()

engine_type          dual_fuel
TU5                  0            1874
توربوشارژ            0            1082
TU3                  0             889
EF7                  0             537
M15                  0             525
                                  ... 
یورو5                0               2
یورو4 (4G69)         0               2
توربوشارژ (9NR-FTS)  0               1
(GAMMA)              0               1
KA24 انژکتوری        0               1
Name: count, Length: 99, dtype: int64

In [61]:
train['dual_fuel'].value_counts()

dual_fuel
0    15322
1      266
Name: count, dtype: int64

# NLP-Based Description Preprocessing

In [62]:
train['description'].isnull().sum()

np.int64(2098)

In [63]:
train['description'] = train['description'].fillna('')

# Text Normalization

In [64]:
train['description'] = train['description'].str.replace('ي', 'ی', regex=False)
train['description'] = train['description'].str.replace('ك', 'ک', regex=False)

# Remove Punctuation

In [65]:
train['description'] = train['description'].str.replace(r'[^\w\s]', ' ', regex=True)

# Whitespace Normalization

In [66]:
train['description'] = train['description'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Tokenization

In [67]:
train['description_tokens'] = train['description'].apply(lambda x: x.split())

# TF_IDF

In [68]:
tfidf = TfidfVectorizer(max_features=500)
description_features = tfidf.fit_transform(train['description'])

In [69]:
description_df = pd.DataFrame(
    description_features.toarray(),
    columns=tfidf.get_feature_names_out(),
    index=train.index
)

# TF_IDF Feature Integration

In [70]:
description_df = description_df.add_prefix('desc_')
train = pd.concat([train, description_df], axis=1)

In [71]:
train.shape

(15588, 522)

# Remove Original Text Columns

In [72]:
train = train.drop(columns = ['description', 'description_tokens'])

# Categorical Features

In [73]:
train.select_dtypes(include = ['object', 'str']).columns

Index(['title', 'brand', 'model', 'transmission', 'body_color', 'inside_color',
       'body_status', 'body_type', 'engine', 'engine_model', 'engine_type',
       'price'],
      dtype='str')

# Remove Unnecessary Columns

In [74]:
train = train.drop(columns = ['title', 'engine', 'engine_type'])

# Categorical Enconding

In [75]:
for col in train.select_dtypes(include = ['object', 'str']).columns:
    print(col, train[col].nunique())

brand 62
model 223
transmission 2
body_color 42
inside_color 41
body_status 15
body_type 7
engine_model 52
price 1298


In [76]:
train = pd.get_dummies(
    train,
    columns=['brand', 'transmission', 'body_status', 'body_type'],
    dtype=int
)

# Price Columns Conversion

In [77]:
train['price'].dtype

<StringDtype(na_value=nan)>

In [78]:
train['price'].head(10)

0    2,550,000,000
1      215,000,000
2      445,000,000
3      310,000,000
4      570,000,000
5    4,990,000,000
6      590,000,000
7      480,000,000
8      390,000,000
9      280,000,000
Name: price, dtype: str

In [79]:
train['price'] = pd.to_numeric(
    train['price'].astype(str).str.replace(',', '', regex=False),
    errors='coerce')
train['price'].head(5)

0    2550000000
1     215000000
2     445000000
3     310000000
4     570000000
Name: price, dtype: int64

# Feature and Target Separation

In [80]:
X = train.drop(columns = ['price'])
y = train['price']

# Train-Test Split

In [81]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

# Enconding & Scaling

In [82]:
categorical_feature = X_train.select_dtypes(include = ['object', 'str']).columns
numerical_feature = X_train.select_dtypes(include = 'number').columns

preprocessor = ColumnTransformer(
    transformers = [
        ('cat', OneHotEncoder(handle_unknown = 'ignore'), categorical_feature),
        ('num', StandardScaler(), numerical_feature)
    ]
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

#   قبل مدل  : چون خطا میداد زدم

In [83]:
# پر کردن مقادیر گمشده ستون‌های عددی با میانه

numeric_columns = ['mileage','fuel','volume','cylinders','acceleration']
for column in numeric_columns:
    train[column] = train[column].fillna(train[column].median())


# پر کردن مقادیر گمشده engine_modelا پرتکرارترین مقدار
train['engine_model'] = train['engine_model'].fillna(train['engine_model'].mode()[0])    


# بررسی مقادیر گمشده بعد از Data Cleaning
missing_values = train.isnull().sum()
missing_values = missing_values[missing_values > 0]
print(missing_values)


# جدا کردن ویژگی‌ها x و متغیر هدف y
X = train.drop(columns=['price'])
y = train['price']

print("X shape:", X.shape)
print("y shape:", y.shape)

Series([], dtype: int64)
X shape: (15588, 598)
y shape: (15588,)


In [84]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (12470, 598)
X_test: (3118, 598)
y_train: (12470,)
y_test: (3118,)


In [85]:
# مشخص کردن ستون‌های عددی و categorical
categorical_features = X_train.select_dtypes(include=['object', 'str']).columns
numerical_features = X_train.select_dtypes(include='number').columns

print("Categorical features:")
print(categorical_features.tolist())

print("\nNumerical features:")
print(numerical_features.tolist())

Categorical features:
['model', 'body_color', 'inside_color', 'engine_model']

Numerical features:
['year', 'mileage', 'fuel', 'volume', 'cylinders', 'turbo', 'dual_fuel', 'acceleration', 'desc_00', 'desc_000', 'desc_01', 'desc_03', 'desc_04', 'desc_05', 'desc_06', 'desc_08', 'desc_080', 'desc_09', 'desc_0918', 'desc_0w20', 'desc_10', 'desc_100', 'desc_1000', 'desc_10000', 'desc_1011', 'desc_106', 'desc_11', 'desc_110', 'desc_111', 'desc_12', 'desc_120', 'desc_13', 'desc_131', 'desc_1394', 'desc_1395', 'desc_1396', 'desc_1398', 'desc_14', 'desc_1400', 'desc_1401', 'desc_1402', 'desc_1403', 'desc_1404', 'desc_1405', 'desc_1406', 'desc_1407', 'desc_1409', 'desc_15', 'desc_150', 'desc_1500', 'desc_151', 'desc_16', 'desc_160', 'desc_1600', 'desc_165', 'desc_1650', 'desc_17', 'desc_175', 'desc_18', 'desc_180', 'desc_1800', 'desc_19', 'desc_193', 'desc_20', 'desc_200', 'desc_2000', 'desc_2008', 'desc_2012', 'desc_2013', 'desc_2014', 'desc_2015', 'desc_2022', 'desc_2023', 'desc_2024', 'desc_2

In [86]:
preprocessor = ColumnTransformer(transformers=[('cat',OneHotEncoder(handle_unknown='ignore'),categorical_features),('num',StandardScaler(),numerical_features)])

# preprocessing روی داده‌های train و test
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (12470, 949)
X_test shape: (3118, 949)


#   LinearRegression

In [87]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_prediction = linear_model.predict(X_test)

linear_r2 = r2_score(y_test, linear_prediction)
linear_mse = mean_squared_error(y_test, linear_prediction)
linear_mae = mean_absolute_error(y_test, linear_prediction)
linear_rmse = np.sqrt(linear_mse)

#   RidgeRegression

In [88]:
ridge_model = Ridge(alpha=1)
ridge_model.fit(X_train, y_train)
ridge_prediction = ridge_model.predict(X_test)

ridge_r2 = r2_score(y_test, ridge_prediction)
ridge_mse = mean_squared_error(y_test,ridge_prediction)
ridge_mae = mean_absolute_error(y_test, ridge_prediction)
ridge_rmse = np.sqrt(ridge_mse)

#   LassoRegression

In [ ]:
lasso_model = Lasso(alpha=1, max_iter=10000)
lasso_model.fit(X_train, y_train)
lasso_prediction = lasso_model.predict(X_test)

lasso_r2 = r2_score(y_test,lasso_prediction)
lasso_mse = mean_squared_error(y_test,lasso_prediction)
lasso_mae = mean_absolute_error(y_test,lasso_prediction)
lasso_rmse = np.sqrt(lasso_mse)

#   Decision Tree

In [ ]:
tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train, y_train)
tree_prediction = tree_model.predict(X_test)

tree_r2 = r2_score(y_test, tree_prediction)
tree_mse = mean_squared_error(y_test, tree_prediction)
tree_mae = mean_absolute_error(y_test, tree_prediction)
tree_tmse = np.sqrt(tree_mse)

#   Random Forest

In [ ]:
rf_model = RandomForestRegression(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_prediction = rf_model.predict(X_test)
rf_r2 = r2_score(y_test, rf_prediction)
rf_mse = mean_squared_error(y_test, rf_prediction)
rf_mae = mean_absolute_error(y_test, rf_prediction)
rf_rmse = np.sqrt(rf_mse)

#   Model Comparison Table

In [ ]:
resulte = pd.DataFrame({
    "model" : [
        "Linear Regression",
        "Ridge Regression",
        "Lasso Regression",
        "Decision Tree",
        "Random Forest"
    ],

    "mse" : [
        linear_mse,
        ridge_mse,
        lasso_mse,
        tree_mse,
        rf_mse
    ],

    "mae" : [
        linear_mae,
        ridge_mae,
        lasso_mae,
        tree_mae,
        rf_mae
    ],

    "rmse" : [
        linear_rmse,
        ridge_rmse,
        lasso_rmse,
        tree_rmse,
        rf_rmse
    ],

    "r2" : [
        linear_r2,
        ridge_r2,
        lasso_r2,
        tree_r2,
        rf_r2
    ]
})

resulte_sort = resulte.sort_values(
    by="rmse",
    ascending=True
)

resulte_sort

#   Finding the best model

In [ ]:
best_by_mse = result.loc[resulte["mse"].indexin()]
